# 2026-07-28 atlas (CyVerse)

Upload next to this notebook (overwrite old copies):
- `postfire_landscape_atlas1.html`
- `build_atlas_cyverse.py`

Tiles are already built. Run **inventory → attach burn map → Open**. Skip the build cell.

The attach cell also rewrites `atlas/index.html` so scenes are **clickable squares on a map** (not just a card list).

In [ ]:
from pathlib import Path

DATE = '2026-07-28'
TILE = 'all'
cwd = Path.cwd().resolve()

def find_mesma_dir():
    checks = []
    for parent in [cwd, *cwd.parents]:
        checks.append(parent / 'new_data' / DATE / 'mesma')
        checks.append(parent / DATE / 'mesma')
    checks.extend([
        Path('/home/jovyan/data-store/new_data') / DATE / 'mesma',
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
    ])
    for p in checks:
        if p.exists():
            return p
    raise FileNotFoundError('Could not find new_data/2026-07-28/mesma')

MESMA = find_mesma_dir()
DATE_DIR = MESMA.parent
NEW = DATE_DIR.parent
TSF = NEW / 'Time_since_fire'

print('MESMA folder:', MESMA)
fracs = []
for tile in sorted(p for p in MESMA.iterdir() if p.is_dir() and p.name != 'test'):
    for c in tile.iterdir():
        if c.is_file() and c.name.endswith('_fractions') and c.suffix == '':
            fracs.append(c)
by_tile = {}
for c in fracs:
    by_tile.setdefault(c.parent.name, c)
selected = [by_tile[k] for k in sorted(by_tile)]
if TILE and TILE != 'all':
    selected = [c for c in selected if TILE in c.parent.name]
print('TILE:', TILE, ' unique tiles:', len(selected))
print('TSF folder:', TSF, 'exists' if TSF.exists() else 'MISSING')
print('Atlas:', DATE_DIR / 'atlas')

## 1. Put the burn map on the website

The tab looks for `southern_california_burn_map.png` **inside each tile folder**. Copying it only to `atlas/` is why the map was blank.

This cell copies the PNG from Time_since_fire into `atlas/` and every tile folder, updates each atlas page, and rebuilds `index.html` so tiles are clickable squares on a street map.

In [ ]:
from pathlib import Path
import shutil

atlas = DATE_DIR / 'atlas'
atlas.mkdir(parents=True, exist_ok=True)

png_name = 'southern_california_burn_map.png'
html_name = 'southern_california_burn_map.html'
search = [
    TSF,
    Path('/home/jovyan/data-store/new_data/Time_since_fire'),
    Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
    Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
    Path.cwd() / 'new_data' / 'Time_since_fire',
    Path.cwd(),
]
print('Looking for', png_name)
for d in search:
    print(f'  png={ (d/png_name).exists() }  html={ (d/html_name).exists() }  {d}')

png = next((d / png_name for d in search if (d / png_name).exists()), None)
if png is None:
    raise FileNotFoundError(
        'southern_california_burn_map.png not found. '
        'It should be in /home/jovyan/data-store/new_data/Time_since_fire/'
    )
print(f'PNG source: {png}  ({png.stat().st_size/1e6:.2f} MB)')

# Tiny page the iframe can load. No Earth Engine, no Leaflet CDN.
viewer = (
    '<!DOCTYPE html><html><head><meta charset="utf-8">'
    '<title>Days since last burn</title>'
    '<style>html,body{margin:0;background:#f4efe4}img{width:100%;height:auto;display:block}</style>'
    '</head><body>'
    f'<img src="{png_name}" alt="Days since last burn">'
    '</body></html>\n'
)

folders = [atlas]
folders.extend(sorted(p for p in atlas.iterdir() if p.is_dir()))
for folder in folders:
    dest_png = folder / png_name
    if png.resolve() != dest_png.resolve():
        shutil.copy2(png, dest_png)
    (folder / html_name).write_text(viewer)

print(f'Wrote PNG + HTML into {len(folders)} folders (atlas + tiles)')

atlas_page = Path.cwd() / 'postfire_landscape_atlas1.html'
if not atlas_page.exists():
    atlas_page = Path('/home/jovyan/data-store/postfire_landscape_atlas1.html')
if atlas_page.exists():
    text = atlas_page.read_text()
    n = 0
    for d in folders:
        if d == atlas:
            continue
        if (d / 'mesma_grid.js').exists():
            (d / 'postfire_landscape_atlas1.html').write_text(text)
            n += 1
    print(f'Updated atlas page in {n} tile folders from {atlas_page}')
else:
    print('postfire_landscape_atlas1.html not next to the notebook — iframe will still show the PNG page')

check = atlas / '11_S_KR_2026_7_28' / png_name
print('example tile PNG exists:', check.exists(), check if check.exists() else '')
print('Run the Open cell, then Cmd+Shift+R on the atlas tab.')


# Rebuild the scene picker so tiles are clickable on a map (no MESMA rebuild).
build_py = Path.cwd() / 'build_atlas_cyverse.py'
if not build_py.exists():
    build_py = Path('/home/jovyan/data-store/build_atlas_cyverse.py')
if build_py.exists():
    import importlib.util
    spec = importlib.util.spec_from_file_location('bac', build_py)
    bac = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(bac)
    tile_folders = [p.name for p in atlas.iterdir() if p.is_dir() and (p / 'mesma_grid.js').exists()]
    bac.write_atlas_index(atlas, tile_folders, mesma_dir=MESMA)
    print('Wrote clickable tile map to', atlas / 'index.html')
else:
    print('build_atlas_cyverse.py not found — upload it next to this notebook to refresh the scene picker')


## 2. Build remaining MESMA tiles

Skip this if tiles already have `mesma_grid.js`. `11_S_PU` has no modeled pixels — that error is expected.

In [ ]:
from IPython import get_ipython

html = Path.cwd() / 'postfire_landscape_atlas1.html'
if not html.exists():
    html = Path('/home/jovyan/data-store/postfire_landscape_atlas1.html')

to_build = []
for frac in selected:
    out = DATE_DIR / 'atlas' / frac.parent.name
    if (out / 'mesma_grid.js').exists():
        print('skip', frac.parent.name)
        continue
    to_build.append(frac)
print(f'Building {len(to_build)} remaining of {len(selected)}')

for frac in to_build:
    out = DATE_DIR / 'atlas' / frac.parent.name
    cmd = (
        f'build_atlas_cyverse.py --fractions {frac} --tsf-dir {TSF} '
        f'--html {html} --out {out}'
    )
    print('\n====', frac.parent.name, '====')
    try:
        get_ipython().run_line_magic('run', cmd)
    except SystemExit as exc:
        print('skipped:', frac.parent.name, exc)

## 3. Open the website

Do not use Jupyter HTML preview. Run this cell and click the link.

In [ ]:
import os, socket, subprocess, time
from IPython.display import display, HTML

atlas = DATE_DIR / 'atlas'
PORT = 8765
pngs = list(atlas.rglob('southern_california_burn_map.png'))
print(f'atlas/ {sum(p.stat().st_size for p in atlas.rglob("*") if p.is_file())/1e6:.1f} MB')
print(f'burn PNG copies: {len(pngs)}')

def port_open(port):
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(PORT):
    subprocess.Popen(
        ['python3', '-m', 'http.server', str(PORT), '--bind', '127.0.0.1'],
        cwd=str(atlas), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(0.8)

prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX') or '/'
if not prefix.endswith('/'):
    prefix += '/'
url = f'{prefix}proxy/{PORT}/index.html'
print(url)
display(HTML(
    f'<p style="font-size:18px"><a href="{url}" target="_blank" rel="noopener">'
    f'<b>Open postfire atlas</b></a></p>'
    f'<p>Hard-refresh (Cmd+Shift+R) after opening.</p>'
))